In [1]:
import os

In [2]:
%pwd

'd:\\PROGRAMMING\\Machine_Learning\\git_text_summerizer\\research'

In [3]:
os.chdir("..")

In [4]:
%pwd

'd:\\PROGRAMMING\\Machine_Learning\\git_text_summerizer'

In [5]:
from dataclasses import dataclass
from pathlib import Path


@dataclass(frozen=True)
class DataPreparationConfig:
    root_dir: Path
    data_path: Path
    train_filename: str
    validation_filename: str
    test_filenames: list[str]
    

In [6]:
from src.ai_model.constants import CONFIG_FILE_PATH, PARAMS_FILE_PATH
from src.ai_model.utils.common import read_yaml, create_directories

In [7]:
class ConfigrationManager:
    def __init__(
        self,
        config_filepath=CONFIG_FILE_PATH,
        params_filepath=PARAMS_FILE_PATH,
    ):
        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)

        create_directories([self.config.artifacts_root])

    def get_data_preparation_config(self) -> DataPreparationConfig:
        config = self.config.data_preparation

        create_directories([config.root_dir])

        data_preparation_config = DataPreparationConfig(
            root_dir=config.root_dir,
            data_path=config.data_path,
            train_filename=config.train_filename,
            validation_filename=config.validation_filename,
            test_filenames=config.test_filenames,
        )

        return data_preparation_config

In [8]:
from datasets import load_dataset, concatenate_datasets
from pathlib import Path


class DataPreparation:
    def __init__(self, config: DataPreparationConfig):
        self.config = config

    def main(self):
        config = self.config

        train_ds_path = Path(f"{config.data_path}/{config.train_filename}")
        validation_ds_path = Path(f"{config.data_path}/{config.validation_filename}")
        test1_ds_path = Path(f"{config.data_path}/{config.test_filenames[0]}")
        test2_ds_path = Path(f"{config.data_path}/{config.test_filenames[1]}")

        dataset = load_dataset(
            "csv",
            data_files={
                "train": str(train_ds_path),
                "validation": str(validation_ds_path),
                "test_1": str(test1_ds_path),
                "test_2": str(test2_ds_path),
            },
        )

        dataset["test"] = concatenate_datasets([dataset["test_1"], dataset["test_2"]])

        dataset.pop("test_1")
        dataset.pop("test_2")
        
        dataset.save_to_disk(config.root_dir)

c:\Users\asus\.conda\envs\text_sum\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [9]:
try:
    config = ConfigrationManager()
    data_preparation_config = config.get_data_preparation_config()
    data_preparation = DataPreparation(config=data_preparation_config)
    data_preparation.main()
except Exception as e:
    raise e

[2026-04-25 13:17:37,846: INFO: common: yaml file: config\config.yaml loaded successfuly]
[2026-04-25 13:17:37,852: INFO: common: yaml file: params.yaml loaded successfuly]
[2026-04-25 13:17:37,855: INFO: common: created directory at: artifacts]
[2026-04-25 13:17:37,856: INFO: common: created directory at: artifacts/data_preparation]
[2026-04-25 13:17:39,457: INFO: _client: HTTP Request: HEAD https://s3.amazonaws.com/datasets.huggingface.co/datasets/datasets/csv/csv.py "HTTP/1.1 200 OK"]


Saving the dataset (1/1 shards): 100%|██████████| 400/400 [00:00<00:00, 38987.77 examples/s]
